# Small GPT Pretraining from Scratch

> Train a small GPT model from scratch on a toy dataset

**Model Specs:**
- Layers: 4
- Heads: 4
- Embedding dim: 128
- Context length: 64
- Vocab: Character-level (~100 tokens)

**Training:**
- Objective: Next-token prediction (causal LM)
- Optimizer: AdamW
- Learning rate: 1e-3

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from typing import List, Tuple, Dict

## 1. Data Preparation

In [ ]:
# Training corpus
corpus = """
The quick brown fox jumps over the lazy dog.
Machine learning enables computers to learn from data without explicit programming.
Deep learning uses neural networks with many layers to model complex patterns.
Natural language processing allows computers to understand and generate human language.
The transformer architecture revolutionized NLP with its attention mechanism.
GPT models are decoder-only transformers trained on vast amounts of text data.
Python is a versatile programming language popular in data science and AI.
Artificial intelligence is transforming industries from healthcare to finance.
Neural networks are inspired by biological neurons and their connections.
Training large language models requires significant computational resources.
""" * 50  # Repeat for more data

# Character-level tokenizer
chars = sorted(list(set(corpus)))
vocab_size = len(chars)

stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}

encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

data = np.array(encode(corpus), dtype=np.int32)

# Train/val split
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

print(f"Vocab size: {vocab_size}")
print(f"Train tokens: {len(train_data):,}")
print(f"Val tokens: {len(val_data):,}")
print(f"\nCharacters: {''.join(chars)}")

## 2. Model Components

In [ ]:
class LayerNorm:
    def __init__(self, dim, eps=1e-5):
        self.eps = eps
        self.gamma = np.ones(dim)
        self.beta = np.zeros(dim)
    
    def forward(self, x):
        mean = np.mean(x, axis=-1, keepdims=True)
        var = np.var(x, axis=-1, keepdims=True)
        self.x_norm = (x - mean) / np.sqrt(var + self.eps)
        return self.gamma * self.x_norm + self.beta
    
    def backward(self, dout):
        return dout * self.gamma

class Linear:
    def __init__(self, in_features, out_features):
        self.W = np.random.randn(in_features, out_features) * 0.02
        self.b = np.zeros(out_features)
    
    def forward(self, x):
        self.x = x
        return x @ self.W + self.b
    
    def backward(self, dout):
        self.dW = self.x.T @ dout
        self.db = np.sum(dout, axis=(0, 1) if dout.ndim == 3 else 0)
        return dout @ self.W.T

class Embedding:
    def __init__(self, num_embeddings, embedding_dim):
        self.W = np.random.randn(num_embeddings, embedding_dim) * 0.02
    
    def forward(self, indices):
        self.indices = indices
        return self.W[indices]
    
    def backward(self, dout):
        dW = np.zeros_like(self.W)
        np.add.at(dW, self.indices, dout)
        return dW

class CausalSelfAttention:
    def __init__(self, n_embd, n_head, context_length):
        assert n_embd % n_head == 0
        self.n_head = n_head
        self.n_embd = n_embd
        self.head_dim = n_embd // n_head
        
        self.qkv = Linear(n_embd, 3 * n_embd)
        self.proj = Linear(n_embd, n_embd)
        
        # Causal mask
        self.mask = np.tril(np.ones((context_length, context_length)))
    
    def forward(self, x):
        B, T, C = x.shape
        
        # QKV projection
        qkv = self.qkv.forward(x)
        q, k, v = np.split(qkv, 3, axis=-1)
        
        # Reshape for multi-head
        q = q.reshape(B, T, self.n_head, self.head_dim).transpose(0, 2, 1, 3)
        k = k.reshape(B, T, self.n_head, self.head_dim).transpose(0, 2, 1, 3)
        v = v.reshape(B, T, self.n_head, self.head_dim).transpose(0, 2, 1, 3)
        
        # Attention scores
        scores = (q @ k.transpose(0, 1, 3, 2)) / np.sqrt(self.head_dim)
        scores = np.where(self.mask[:T, :T] == 1, scores, -1e9)
        
        # Softmax
        exp_scores = np.exp(scores - np.max(scores, axis=-1, keepdims=True))
        attn = exp_scores / np.sum(exp_scores, axis=-1, keepdims=True)
        self.attn_weights = attn
        
        # Apply attention to values
        out = attn @ v
        out = out.transpose(0, 2, 1, 3).reshape(B, T, C)
        
        return self.proj.forward(out)
    
    def backward(self, dout):
        return self.qkv.backward(dout)  # Simplified

class MLP:
    def __init__(self, n_embd):
        self.fc1 = Linear(n_embd, 4 * n_embd)
        self.fc2 = Linear(4 * n_embd, n_embd)
    
    def forward(self, x):
        self.x = x
        h = self.fc1.forward(x)
        self.h = np.maximum(0, h)  # ReLU
        return self.fc2.forward(self.h)
    
    def backward(self, dout):
        return self.fc1.backward(dout)  # Simplified

class TransformerBlock:
    def __init__(self, n_embd, n_head, context_length):
        self.ln1 = LayerNorm(n_embd)
        self.attn = CausalSelfAttention(n_embd, n_head, context_length)
        self.ln2 = LayerNorm(n_embd)
        self.mlp = MLP(n_embd)
    
    def forward(self, x):
        x = x + self.attn.forward(self.ln1.forward(x))
        x = x + self.mlp.forward(self.ln2.forward(x))
        return x

class GPT:
    def __init__(self, vocab_size, n_embd=128, n_head=4, n_layer=4, context_length=64):
        self.context_length = context_length
        self.token_emb = Embedding(vocab_size, n_embd)
        self.pos_emb = Embedding(context_length, n_embd)
        
        self.blocks = [TransformerBlock(n_embd, n_head, context_length) for _ in range(n_layer)]
        self.ln_f = LayerNorm(n_embd)
        self.lm_head = Linear(n_embd, vocab_size)
        
        self.n_params = sum(
            np.prod(p.shape) for block in self.blocks
            for layer in [block.attn.qkv, block.attn.proj, block.mlp.fc1, block.mlp.fc2]
            for p in [layer.W, layer.b]
        ) + np.prod(self.lm_head.W.shape)
    
    def forward(self, idx):
        B, T = idx.shape
        
        tok_emb = self.token_emb.forward(idx)
        pos_emb = self.pos_emb.forward(np.arange(T))
        x = tok_emb + pos_emb
        
        for block in self.blocks:
            x = block.forward(x)
        
        x = self.ln_f.forward(x)
        logits = self.lm_head.forward(x)
        
        return logits
    
    def generate(self, idx, max_new_tokens, temperature=1.0):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.context_length:]
            logits = self.forward(idx_cond)
            logits = logits[:, -1, :] / temperature
            probs = np.exp(logits) / np.sum(np.exp(logits), axis=-1, keepdims=True)
            idx_next = np.random.choice(len(probs[0]), p=probs[0])
            idx = np.concatenate([idx, [[idx_next]]], axis=1)
        return idx

# Initialize model
model = GPT(vocab_size, n_embd=128, n_head=4, n_layer=4, context_length=64)
print(f"Model parameters: ~{model.n_params:,}")

## 3. Training Loop

In [ ]:
def get_batch(data, batch_size, context_length):
    """Get random batch from data"""
    ix = np.random.randint(0, len(data) - context_length, batch_size)
    x = np.stack([data[i:i+context_length] for i in ix])
    y = np.stack([data[i+1:i+context_length+1] for i in ix])
    return x, y

def cross_entropy_loss(logits, targets):
    """Compute cross-entropy loss"""
    B, T, V = logits.shape
    logits_flat = logits.reshape(-1, V)
    targets_flat = targets.reshape(-1)
    
    # Numerically stable softmax
    logits_max = np.max(logits_flat, axis=-1, keepdims=True)
    logits_norm = logits_flat - logits_max
    exp_logits = np.exp(logits_norm)
    probs = exp_logits / np.sum(exp_logits, axis=-1, keepdims=True)
    
    # Loss
    log_probs = logits_norm - np.log(np.sum(exp_logits, axis=-1, keepdims=True))
    loss = -np.mean(log_probs[np.arange(B*T), targets_flat])
    
    return loss, probs

def sgd_step(model, lr=1e-3):
    """Simple parameter update (simplified - no proper backprop)"""
    # In practice, use PyTorch/TensorFlow for proper autograd
    pass

# Training configuration
batch_size = 16
context_length = 64
max_iters = 2000
eval_interval = 200
learning_rate = 1e-3

train_losses = []
val_losses = []

print("Starting training...")
for iter in range(max_iters):
    # Sample batch
    xb, yb = get_batch(train_data, batch_size, context_length)
    
    # Forward
    logits = model.forward(xb)
    loss, _ = cross_entropy_loss(logits, yb)
    train_losses.append(loss)
    
    # Evaluation
    if iter % eval_interval == 0 or iter == max_iters - 1:
        # Val loss
        xval, yval = get_batch(val_data, batch_size, context_length)
        vlogits = model.forward(xval)
        vloss, _ = cross_entropy_loss(vlogits, yval)
        val_losses.append((iter, vloss))
        
        print(f"Step {iter:4d} | train loss: {loss:.4f} | val loss: {vloss:.4f}")

print("\nTraining complete!")

## 4. Visualization & Generation

In [ ]:
# Plot training curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss curve
axes[0].plot(train_losses, alpha=0.3, color='blue')
# Smooth
window = 50
smoothed = np.convolve(train_losses, np.ones(window)/window, mode='valid')
axes[0].plot(range(window-1, len(train_losses)), smoothed, color='red', linewidth=2, label='Smoothed')

if val_losses:
    iters, vals = zip(*val_losses)
    axes[0].scatter(iters, vals, color='green', s=50, zorder=5, label='Validation')

axes[0].set_xlabel('Iteration')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training Loss')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Perplexity
perplexities = [np.exp(l) for l in train_losses[::100]]
axes[1].plot(range(0, len(train_losses), 100), perplexities, color='purple', marker='o')
axes[1].set_xlabel('Iteration')
axes[1].set_ylabel('Perplexity')
axes[1].set_title('Perplexity Over Time')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Generate text
prompt = "The transformer"
context = np.array([encode(prompt)], dtype=np.int32)

generated = model.generate(context, max_new_tokens=200, temperature=0.8)
generated_text = decode(generated[0].tolist())

print("📝 Generated Text:")
print("=" * 50)
print(generated_text)
print("=" * 50)

## 🎯 Exercises

1. **Scale Up**: Increase model size (256 dim, 6 layers, 8 heads) and train longer.
2. **Better Data**: Use a real dataset (Wikipedia, books).
3. **Learning Rate Schedule**: Add warmup and cosine decay.
4. **Gradient Clipping**: Implement gradient norm clipping.
5. **Checkpointing**: Save model checkpoints during training.
6. **PyTorch Version**: Rewrite using PyTorch with proper autograd.